# Heart Failure Prediction Models Comparison

This notebook combines 10 different machine learning and deep learning models to predict heart failure survival.
It includes:
1. Data Loading & Preprocessing
2. PyTorch Model Definitions (MLP, CNN, Transformer, LSTM)
3. SKLearn Model Definitions (Logistic Regression, RF, SVM, KNN, XGBoost, Ensemble)
4. Training & Evaluation
5. SHAP Explainability
6. Comparison Plots

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, roc_curve, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.inspection import permutation_importance
import xgboost as xgb
import shap
import os
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# Ensure reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 1. Data Loading & Preprocessing

In [ ]:
def load_and_preprocess_data(filepath):
    print("Loading data...")
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        # Fallback for Colab if file is not uploaded, typically user needs to upload it
        print(f"File '{filepath}' not found. Please ensure it is uploaded.")
        return None, None, None, None, None, None, None, None

    features = ['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes', 
                'ejection_fraction', 'high_blood_pressure', 'platelets', 
                'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time']
    target = 'DEATH_EVENT'
    
    X = df[features]
    y = df[target]
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    # Scale
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=features)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=features)
    
    return df, X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled, scaler

## 2. PyTorch Model Classes

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super(MLP, self).__init__()
        self.layer1 = nn.Linear(input_dim, 32)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.layer2 = nn.Linear(32, 16)
        self.layer3 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.dropout(x)
        x = self.relu(self.layer2(x))
        x = self.sigmoid(self.layer3(x))
        return x

class CNN1D(nn.Module):
    def __init__(self, num_features):
        super(CNN1D, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(16 * (num_features // 2), 32)
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.relu(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        return x

class TabularTransformer(nn.Module):
    def __init__(self, input_dim, d_model=32, nhead=2):
        super(TabularTransformer, self).__init__()
        self.embedding = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.fc = nn.Linear(d_model, 1)
        self.dropout = nn.Dropout(0.2)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        # x shape: [Batch, 1, Features]
        x = self.embedding(x) # [Batch, 1, d_model]
        x = self.transformer_encoder(x)
        x = x.mean(dim=1) # Pooling over sequence (length 1)
        x = self.dropout(x)
        x = self.sigmoid(self.fc(x))
        return x

class HybridLSTM(nn.Module):
    def __init__(self, seq_input_dim, static_input_dim, hidden_dim):
        super(HybridLSTM, self).__init__()
        self.lstm = nn.LSTM(seq_input_dim, hidden_dim, batch_first=True)
        self.fc_static = nn.Linear(static_input_dim, 16)
        self.fc_final = nn.Linear(hidden_dim + 16, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x_seq, x_static):
        lstm_out, (hn, cn) = self.lstm(x_seq)
        last_hidden = hn[-1]
        static_out = self.relu(self.fc_static(x_static))
        combined = torch.cat((last_hidden, static_out), 1)
        combined = self.dropout(combined)
        out = self.sigmoid(self.fc_final(combined))
        return out

In [ ]:
# Helper for LSTM data simulation
def simulate_vitals(row, seq_len=24):
    np.random.seed(42) # Reproducibility within function
    age_factor = row['age'] / 100
    base_hr = 70 + (row['anaemia'] * 10) + np.random.normal(0, 5)
    hr = np.random.normal(base_hr, 5, seq_len)
    base_bp = 110 + (row['high_blood_pressure'] * 20) + np.random.normal(0, 5)
    bp = np.random.normal(base_bp, 10, seq_len)
    return np.stack([hr, bp], axis=1)

def prepare_lstm_data(df, X_scaled):
    vitals_list = []
    # We iterate over the dataframe to simulate vitals based on original features
    # Note: X_scaled might be passed in some contexts, but here we use df if available or assume X_scaled structure
    # To be safe and consistent with script logic:
    if isinstance(df, pd.DataFrame):
        iterator = df.iterrows()
    else:
        # Fallback if df is actually numpy array or tensor (unlikely given usage below)
        return None, None 
        
    for idx, row in iterator:
        vitals_list.append(simulate_vitals(row))
    X_seq = np.array(vitals_list)
    return torch.FloatTensor(X_seq), torch.FloatTensor(X_scaled.values)

## 3. Training & Wrappers

In [ ]:
class TorchWrapper:
    def __init__(self, model, model_type='standard', feature_names=None):
        self.model = model
        self.model.eval()
        self.model_type = model_type
        self.feature_names = feature_names
        
    def predict_proba(self, X):
        # X is numpy array [N, features]
        with torch.no_grad():
            tensor_X = torch.FloatTensor(X)
            
            if self.model_type == 'cnn':
                tensor_X = tensor_X.unsqueeze(1)
            elif self.model_type == 'transformer':
                tensor_X = tensor_X.unsqueeze(1)
            elif self.model_type == 'lstm':
                N = X.shape[0]
                seq_len = 24
                # Random noise simulation for SHAP/black-box explanation where original DF isn't available
                X_seq = torch.randn(N, seq_len, 2) 
                tensor_X_static = tensor_X
                out = self.model(X_seq, tensor_X_static)
                prob = out.numpy().flatten()
                return np.stack([1-prob, prob], axis=1)

            out = self.model(tensor_X)
            prob = out.numpy().flatten()
            return np.stack([1-prob, prob], axis=1)

def train_sklearn_model(name, model, X_train, y_train):
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    return model

def train_torch_model(name, model, X_train, y_train, epochs=150, lr=0.001, model_type='standard'):
    print(f"Training {name} (PyTorch)...")
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    y_train_t = torch.FloatTensor(y_train.values).reshape(-1, 1)
    
    losses = []
    
    if model_type == 'lstm':
        # For LSTM training, we use the prepared sequences
        # We assume X_train passed here is the SCALED dataframe, which we use to generate vitals
        X_seq_t, X_static_t = prepare_lstm_data(X_train, X_train)
        
        for epoch in range(epochs):
            model.train()
            optimizer.zero_grad()
            outputs = model(X_seq_t, X_static_t)
            loss = criterion(outputs, y_train_t)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            
    else:
        # Standard, CNN, MLP, Transformer
        X_train_t = torch.FloatTensor(X_train.values)
        if model_type in ['cnn', 'transformer']:
            X_train_t = X_train_t.unsqueeze(1)
            
        for epoch in range(epochs):
            model.train()
            optimizer.zero_grad()
            outputs = model(X_train_t)
            loss = criterion(outputs, y_train_t)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

    return model

## 4. Evaluation & SHAP

In [ ]:
def evaluate_model(model, X_test, y_test, model_type='sklearn', scaler=None):
    if model_type == 'sklearn':
        y_pred = model.predict(X_test)
        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(X_test)[:, 1]
        else: 
            y_prob = model.decision_function(X_test)
            
    elif model_type in ['cnn', 'transformer', 'standard', 'mlp']:
        model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X_test.values)
            if model_type in ['cnn', 'transformer']:
                X_tensor = X_tensor.unsqueeze(1)
            y_prob = model(X_tensor).numpy().flatten()
            y_pred = (y_prob > 0.5).astype(int)
            
    elif model_type == 'lstm':
        model.eval()
        with torch.no_grad():
            # Generate seq data for test
            X_seq_t, X_static_t = prepare_lstm_data(X_test, X_test)
            y_prob = model(X_seq_t, X_static_t).numpy().flatten()
            y_pred = (y_prob > 0.5).astype(int)

    acc = accuracy_score(y_test, y_pred)
    try:
        auc = roc_auc_score(y_test, y_prob)
    except:
        auc = 0.5
        
    return acc, auc, y_pred, y_prob

def explain_model_shap(name, model, X_train, X_test, model_type='sklearn'):
    print(f"Generating SHAP for {name}...")
    
    # Select Explainer
    if name in ['Logistic Regression']:
        explainer = shap.LinearExplainer(model, X_train)
        shap_values = explainer.shap_values(X_test)
        
    elif name in ['Random Forest', 'XGBoost']:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_test)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
            
    elif model_type in ['cnn', 'transformer', 'mlp', 'standard', 'lstm']:
        wrapper = TorchWrapper(model, model_type=model_type, feature_names=X_test.columns.tolist())
        background = shap.kmeans(X_train.values, 10) 
        explainer = shap.KernelExplainer(wrapper.predict_proba, background)
        shap_values = explainer.shap_values(X_test.values, nsamples=100)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]

    else:
        if hasattr(model, "predict_proba"):
            f = model.predict_proba
        else:
            f = model.decision_function
            
        background = shap.kmeans(X_train, 10)
        explainer = shap.KernelExplainer(f, background)
        shap_values = explainer.shap_values(X_test, nsamples=100)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
            
    # Visualize
    plt.figure()
    shap.summary_plot(shap_values, X_test, show=False)
    plt.title(f'SHAP Summary - {name}')
    plt.tight_layout()
    plt.show() # Changed from savefig to show for notebook
    
    return shap_values

## 5. Main Execution and Comparison

In [ ]:
# Load Data
filename = 'cardiac_failure_data.csv'
df, X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled, scaler = load_and_preprocess_data(filename)

if df is not None:
    # Define Models
    models = {}
    
    # 1. Logistic Regression
    models['Logistic Regression'] = LogisticRegression(random_state=42, solver='liblinear', class_weight='balanced')
    
    # 2. Random Forest
    models['Random Forest'] = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, class_weight='balanced')
    
    # 3. SVM
    models['SVM'] = SVC(kernel='rbf', C=1.0, probability=True, class_weight='balanced', random_state=42)
    
    # 4. KNN
    models['KNN'] = KNeighborsClassifier(n_neighbors=7)
    
    # 5. XGBoost
    models['XGBoost'] = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    
    # 6. Ensemble Voting
    models['Ensemble'] = VotingClassifier(
        estimators=[
            ('lr', LogisticRegression(random_state=42)),
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('svc', SVC(probability=True, random_state=42)),
            ('xgb', xgb.XGBClassifier(eval_metric='logloss', use_label_encoder=False, random_state=42))
        ], voting='soft'
    )
    
    # PyTorch Models
    mlp = MLP(input_dim=12)
    cnn = CNN1D(num_features=12)
    tabnet = TabularTransformer(input_dim=12)
    lstm = HybridLSTM(seq_input_dim=2, static_input_dim=12, hidden_dim=32)
    
    results = {}
    
    print("--- Training SKLearn Models ---")
    for name, model in models.items():
        print(f"\nProcessing {name}...")
        try:
            mod = train_sklearn_model(name, model, X_train_scaled, y_train)
            acc, auc, _, y_prob = evaluate_model(mod, X_test_scaled, y_test, model_type='sklearn')
            results[name] = {'Accuracy': acc, 'AUC': auc, 'Prob': y_prob}
            explain_model_shap(name, mod, X_train_scaled, X_test_scaled, model_type='sklearn')
        except Exception as e:
            print(f"Error in {name}: {e}")
            import traceback
            traceback.print_exc()

    print("\n--- Training PyTorch Models ---")
    torch_models = {
        'MLP': (mlp, 'mlp'),
        '1D CNN': (cnn, 'cnn'),
        'TabNet': (tabnet, 'transformer'),
        'LSTM': (lstm, 'lstm')
    }
    
    for name, (model, m_type) in torch_models.items():
        print(f"\nProcessing {name}...")
        try:
            mod = train_torch_model(name, model, X_train_scaled, y_train, epochs=20, model_type=m_type)
            acc, auc, _, y_prob = evaluate_model(mod, X_test_scaled, y_test, model_type=m_type)
            results[name] = {'Accuracy': acc, 'AUC': auc, 'Prob': y_prob}
            explain_model_shap(name, mod, X_train_scaled, X_test_scaled, model_type=m_type)
        except Exception as e:
            print(f"Error in {name}: {e}")
            import traceback
            traceback.print_exc()

    # --- COMPARISON PLOTS ---
    print("\nGenerating Comparison Plots...")
    
    # ROC Curves
    plt.figure(figsize=(10, 8))
    for name, res in results.items():
        if 'Prob' in res and res['Prob'] is not None:
            fpr, tpr, _ = roc_curve(y_test, res['Prob'])
            plt.plot(fpr, tpr, label=f"{name} (AUC = {res['AUC']:.2f})")
    
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves - Model Comparison')
    plt.legend()
    plt.show() # Changed to show
    
    # Metric Comparison
    metric_df = pd.DataFrame(results).T[['Accuracy', 'AUC']]
    metric_df.plot(kind='bar', figsize=(12, 6))
    plt.title('Model Performance Comparison')
    plt.ylabel('Score')
    plt.tight_layout()
    plt.show() # Changed to show
    
    print("\nDone!")
else:
    print("Data loading failed. Please upload 'cardiac_failure_data.csv'.")